# 🏦 Model Risk Validation — Automated Data Quality Assessment Framework
### 13-Dimension EDA Engine · v2.0

---

> **How to use:**
> 1. Edit only the `⚙️ CONFIGURATION` cell — set `DATASET_PATH`
> 2. Optionally declare `ID_COLUMNS`, `DATE_COLUMNS`, `EXPECTED_CATEGORIES`
> 3. Run → `Kernel → Restart & Run All`
> 4. PDF + Excel reports auto-save to your working directory

---

| # | Dimension | Scope | Core Tests |
|---|-----------|-------|------------|
| 1 | **Accuracy** | Values reflect true reality | Z-score, IQR, Isolation Forest |
| 2 | **Availability** | Data present & accessible | Null rate, fill-rate |
| 3 | **Completeness** | No missing required values | Missing %, MCAR proxy |
| 4 | **Comprehensiveness** | Full coverage | Coverage ratio, temporal gap |
| 5 | **Consistency** | Internal coherence | CV, encoding uniformity |
| 6 | **Plausibility** | Values make sense | Tukey fences, domain bounds |
| 7 | **Reasonableness** | Within expected range | IQR, percentile bounds |
| 8 | **Replicability** | Reproducible results | SHA-256 hash |
| 9 | **Representativeness** | Sample ≈ population | Shapiro-Wilk, skew/kurtosis |
| 10 | **Timeliness** | Data is current | Recency analysis |
| 11 | **Traceability** | Lineage trackable | Key column / ID audit |
| 12 | **Uniqueness** | No duplicates | Dup rows, cardinality |
| 13 | **Validity** | Conforms to format/rules | Type & format checks |

In [ ]:
# ── Install / verify dependencies ─────────────────────────────────────────────
import subprocess, sys, importlib

REQUIRED = {
    "fpdf":     "fpdf2",
    "scipy":    "scipy",
    "sklearn":  "scikit-learn",
    "seaborn":  "seaborn",
    "openpyxl": "openpyxl",
}
for module, pkg in REQUIRED.items():
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ All dependencies ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║            ⚙️  CONFIGURATION — EDIT THIS CELL ONLY                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

DATASET_PATH   = "your_dataset.csv"      # ← YOUR CSV PATH HERE

REPORT_TITLE   = "Model Risk Data Quality Assessment"
REPORT_SUBTITLE= "SR 11-7 / OCC Model Risk Management Framework"
REPORT_AUTHOR  = "Model Validation Team"
MODEL_NAME     = "Model Name / ID"       # e.g., "PD Model v3.2"

OUTPUT_PDF     = "DQ_Validation_Report.pdf"
OUTPUT_EXCEL   = "DQ_Validation_Results.xlsx"

# Optional: known ID columns (traceability & uniqueness checks)
ID_COLUMNS     = []   # e.g., ["loan_id", "customer_id"]

# Optional: known date columns (timeliness checks)
DATE_COLUMNS   = []   # e.g., ["origination_date", "report_date"]

# Optional: expected categories per categorical column
EXPECTED_CATEGORIES = {}
# e.g., {"status": ["Active","Closed","Default"], "region": ["N","S","E","W"]}

# Optional: expected numeric ranges per column
EXPECTED_RANGES = {}
# e.g., {"age": (18, 100), "ltv": (0, 2), "credit_score": (300, 850)}

# ── Thresholds ────────────────────────────────────────────────────────────────
MISSING_WARN    = 0.05   # >5%  missing  → WARN
MISSING_FAIL    = 0.20   # >20% missing  → FAIL
OUTLIER_WARN    = 0.02   # >2%  outliers → WARN
OUTLIER_FAIL    = 0.05   # >5%  outliers → FAIL
DUPLICATE_WARN  = 0.01   # >1%  dup rows → WARN
DUPLICATE_FAIL  = 0.05   # >5%  dup rows → FAIL
CARDINALITY_HIGH= 0.95   # >95% unique   → candidate key
SKEW_WARN       = 1.0    # |skew| > 1   → WARN
SKEW_FAIL       = 2.0    # |skew| > 2   → FAIL
STALENESS_WARN  = 180    # days          → WARN
STALENESS_FAIL  = 365    # days          → FAIL

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, kstest, chi2_contingency,
                          entropy as scipy_entropy,
                          skew as sp_skew, kurtosis as sp_kurtosis,
                          probplot)
from sklearn.ensemble import IsolationForest
import warnings, hashlib, datetime, os, re, json, textwrap
from IPython.display import display, Markdown, HTML
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
warnings.filterwarnings("ignore")

# ── Plot style ─────────────────────────────────────────────────────────────────
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except:
    try:
        plt.style.use("seaborn-whitegrid")
    except:
        pass

PALETTE = {
    "PASS":"#27AE60","WARN":"#E67E22","FAIL":"#E74C3C",
    "NA":"#95A5A6","INFO":"#3498DB","dark":"#2C3E50",
    "light":"#ECF0F1","accent":"#8E44AD","bg":"#FAFAFA","border":"#BDC3C7"
}
EMOJI = {"PASS":"✅","WARN":"⚠️","FAIL":"❌","NA":"➖"}
matplotlib.rcParams.update({
    "font.family":"DejaVu Sans","font.size":10,
    "axes.titlesize":12,"axes.labelsize":10,
    "figure.facecolor":PALETTE["bg"],"axes.facecolor":"white"
})
print("✅ Imports complete.")

In [ ]:
# ── Load & Profile Dataset ──────────────────────────────────────────────────────
try:
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    print(f"✅ Loaded: {DATASET_PATH}")
except FileNotFoundError:
    print("⚠️  File not found — generating DEMO synthetic financial dataset...")
    np.random.seed(42)
    n = 1200
    df_raw = pd.DataFrame({
        "loan_id":          [f"LN{i:06d}" for i in range(n)],
        "origination_date": pd.date_range("2020-01-01", periods=n, freq="D")
                              .strftime("%Y-%m-%d").tolist(),
        "age":              np.random.normal(42,12,n).clip(18,85).round(0),
        "income":           np.random.lognormal(10.5,0.6,n).round(2),
        "credit_score":     np.random.normal(680,80,n).clip(300,850).round(0),
        "loan_amount":      np.random.lognormal(11,0.8,n).round(2),
        "ltv":              np.random.beta(4,3,n).round(4),
        "status":           np.random.choice(["Active","Closed","Default","Prepaid"],
                                             n, p=[0.6,0.2,0.12,0.08]),
        "region":           np.random.choice(["North","South","East","West"],n),
        "product_type":     np.random.choice(["Home","Auto","Personal","Business"],
                                             n, p=[0.4,0.3,0.2,0.1]),
        "delinquency_flag": np.random.choice([0,1],n,p=[0.88,0.12]),
        "interest_rate":    np.random.normal(7.5,1.8,n).clip(1,25).round(3),
    })
    # Inject realistic quality issues
    for col in ["age","income","credit_score","interest_rate"]:
        idx = np.random.choice(n, int(n*0.04), replace=False)
        df_raw.loc[idx, col] = np.nan
    dup_idx = np.random.choice(n, 15, replace=False)
    df_raw  = pd.concat([df_raw, df_raw.iloc[dup_idx]], ignore_index=True)
    df_raw.loc[np.random.choice(len(df_raw),8), "income"] *= 50    # outliers
    df_raw.loc[np.random.choice(len(df_raw),5), "age"]    = 150    # implausible

df = df_raw.copy()

# ── Auto-detect & convert date columns ────────────────────────────────────────
DATE_HINTS = ["date","time","dt","year","month","day","period","vintage","asof"]
for col in df.columns:
    if col in DATE_COLUMNS or any(h in col.lower() for h in DATE_HINTS):
        try:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            if col not in DATE_COLUMNS:
                DATE_COLUMNS.append(col)
        except:
            pass

# ── Infer column types ─────────────────────────────────────────────────────────
def infer_col_type(series):
    if pd.api.types.is_datetime64_any_dtype(series): return "datetime"
    if pd.api.types.is_numeric_dtype(series):        return "numeric"
    if series.dtype == object:
        nuniq = series.nunique()
        ratio = nuniq / max(series.count(), 1)
        if ratio > 0.8 and nuniq > 50:               return "text"
    return "categorical"

COL_TYPES = {c: infer_col_type(df[c]) for c in df.columns}
NROWS, NCOLS = df.shape
DATASET_HASH  = hashlib.sha256(
    pd.util.hash_pandas_object(df).values.tobytes()).hexdigest()
LOAD_TIMESTAMP = datetime.datetime.now()

print(f"\n📊 Dataset Profile")
print(f"   Rows     : {NROWS:,}")
print(f"   Columns  : {NCOLS}")
print(f"   Memory   : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
print(f"   SHA-256  : {DATASET_HASH[:32]}...")
print(f"\n🔖 Column Types:")
for col, t in COL_TYPES.items():
    print(f"   {col:<30} → {t}")

In [ ]:
# ── Dataset Column Summary ─────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Type":    [COL_TYPES[c] for c in df.columns],
    "Non-Null": df.count(),
    "Null":    df.isnull().sum(),
    "Null%":   (df.isnull().mean()*100).round(2),
    "Unique":  df.nunique(),
    "Unique%": (df.nunique()/NROWS*100).round(2),
})
display(Markdown("## 📋 Dataset Column Summary"))
display(summary.style
    .background_gradient(subset=["Null%"],   cmap="Reds")
    .background_gradient(subset=["Unique%"], cmap="Blues")
    .format({"Null%":"{:.1f}%","Unique%":"{:.1f}%"})
    .set_caption(f"Shape: {NROWS:,} × {NCOLS}  |  {DATASET_PATH}"))

In [ ]:
# ── Dimension Definitions & Feasibility Matrix ────────────────────────────────
DIMENSIONS = {
    "Accuracy": {
        "definition": (
            "Degree to which values correctly represent real-world attributes. "
            "Inaccurate training data propagates error directly into model coefficients."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Free-text fields lack ground truth for automated accuracy checks."}
    },
    "Availability": {
        "definition": (
            "Proportion of expected records and fields that are present and accessible. "
            "Non-random missingness (MNAR) biases model training and reduces effective N."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
    "Completeness": {
        "definition": (
            "No required values are absent. Goes beyond availability — a field may exist "
            "but be systematically blank. Critical for fields where missingness triggers "
            "imputation or record exclusion."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
    "Comprehensiveness": {
        "definition": (
            "All expected values, categories, and time periods are represented. "
            "A comprehensive dataset covers the full input space the model encounters "
            "in production, including rare but important segments."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Requires domain-specific vocabulary lists for text fields."}
    },
    "Consistency": {
        "definition": (
            "Values are internally coherent — no contradictions within or across fields. "
            "E.g., age derived from DOB must match the age field; categorical codes must "
            "follow a single encoding convention throughout."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Consistency for free text requires semantic NLP analysis."}
    },
    "Plausibility": {
        "definition": (
            "Values are believable within their domain context, even if not provably accurate. "
            "Implausible values (e.g., age=200, LTV=50) suggest data entry errors, system "
            "faults, or unit mismatches."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Plausibility for unstructured text is subjective."}
    },
    "Reasonableness": {
        "definition": (
            "Statistical distribution of values is within expected ranges based on the "
            "nature of the variable. Unlike plausibility (single values), reasonableness "
            "examines the aggregate distributional shape."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Cannot assess without structured value semantics."}
    },
    "Replicability": {
        "definition": (
            "The dataset can be reproduced from source systems and yields the same results "
            "when processed identically. Assessed via cryptographic hash to detect upstream "
            "changes between validation runs."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
    "Representativeness": {
        "definition": (
            "The sample distribution accurately reflects the target population the model "
            "will be applied to. Biased sampling causes poor OOS performance and potential "
            "model discrimination."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":False},
        "infeasible_reason": {"text":"Requires population corpus comparison for text fields."}
    },
    "Timeliness": {
        "definition": (
            "Data reflects the state of the world at the relevant point in time. "
            "Stale data may capture a different economic/behavioral regime than the model's "
            "deployment environment, violating temporal stationarity."),
        "feasible": {"numeric":False,"categorical":False,"datetime":True,"text":False},
        "infeasible_reason": {
            "numeric":     "Timeliness is inferred from datetime columns only.",
            "categorical": "Timeliness is inferred from datetime columns only.",
            "text":        "Timeliness requires a timestamp reference."
        }
    },
    "Traceability": {
        "definition": (
            "Data lineage is documentable — origin system, extraction logic, and "
            "transformation steps are identifiable. In automated assessment, we check "
            "for unique identifiers enabling record-level lineage."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
    "Uniqueness": {
        "definition": (
            "No record appears more than once. Duplicate records inflate effective sample "
            "size, bias frequency estimates, and can cause data leakage in train/test splits."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
    "Validity": {
        "definition": (
            "Values conform to their defined domain — correct data type, valid format, "
            "and within allowable value sets. Invalid values cause silent downstream "
            "errors in model pipelines."),
        "feasible": {"numeric":True,"categorical":True,"datetime":True,"text":True},
        "infeasible_reason": {}
    },
}

display(Markdown("## 📖 Dimension Definitions & Feasibility Matrix"))
rows = []
for dim, info in DIMENSIONS.items():
    f = info["feasible"]
    rows.append(
        f"| **{dim}** | {'✅' if f.get('numeric') else '➖'} | "
        f"{'✅' if f.get('categorical') else '➖'} | "
        f"{'✅' if f.get('datetime') else '➖'} | "
        f"{'✅' if f.get('text') else '➖'} |"
    )
table = (
    "| Dimension | Numeric | Categorical | Datetime | Text |\n"
    "|-----------|:-------:|:-----------:|:--------:|:----:|\n"
    + "\n".join(rows)
)
display(Markdown(table))

In [ ]:
# ── DQResult class & helpers ───────────────────────────────────────────────────
class DQResult:
    """Stores a single dimension × column assessment result."""
    def __init__(self, dimension, column, col_type, feasible,
                 rating="NA", stats=None, findings="", recommendation="",
                 infeasible_reason=""):
        self.dimension         = dimension
        self.column            = column
        self.col_type          = col_type
        self.feasible          = feasible
        self.rating            = rating if feasible else "NA"
        self.stats             = stats or {}
        self.findings          = findings
        self.recommendation    = recommendation
        self.infeasible_reason = infeasible_reason

    def to_dict(self):
        d = {"Dimension":self.dimension,"Column":self.column,
             "Type":self.col_type,"Feasible":self.feasible,
             "Rating":self.rating,"Findings":self.findings,
             "Recommendation":self.recommendation,
             "Infeasible_Reason":self.infeasible_reason}
        d.update({f"stat_{k}":v for k,v in self.stats.items()})
        return d


def rate(val, warn_t, fail_t, higher_is_worse=True):
    """Return PASS/WARN/FAIL rating based on thresholds."""
    if higher_is_worse:
        return "FAIL" if val >= fail_t else ("WARN" if val >= warn_t else "PASS")
    else:
        return "FAIL" if val <= fail_t else ("WARN" if val <= warn_t else "PASS")


def feasibility_check(dimension, col_type):
    info = DIMENSIONS[dimension]
    ok   = info["feasible"].get(col_type, False)
    why  = info["infeasible_reason"].get(col_type,
           "Not applicable for this data type.")
    return ok, why


def norm_test(series):
    """Returns (stat, p, test_name). Chooses test by sample size."""
    s = series.dropna()
    n = len(s)
    if n < 8:
        return None, None, "insufficient_n"
    if n <= 5000:
        stat, p = shapiro(s)
        return round(float(stat),5), round(float(p),5), "Shapiro-Wilk"
    stat, p = kstest(s, "norm", args=(s.mean(), s.std()))
    return round(float(stat),5), round(float(p),5), "Kolmogorov-Smirnov"


ALL_RESULTS = []
print("✅ DQResult class and helpers ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 13 ASSESSMENT FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

# ── 1. ACCURACY ───────────────────────────────────────────────────────────────
def assess_accuracy(df, col, ct):
    ok, why = feasibility_check("Accuracy", ct)
    if not ok:
        return DQResult("Accuracy",col,ct,False,infeasible_reason=why)
    s  = df[col].dropna(); n = len(s)
    st = {}
    if ct == "numeric":
        z     = np.abs(stats.zscore(s))
        out_z = (z > 3).sum()
        Q1, Q3= s.quantile(.25), s.quantile(.75)
        IQR   = Q3-Q1
        out_iq= ((s<Q1-1.5*IQR)|(s>Q3+1.5*IQR)).sum()
        out_if= 0
        if n >= 20:
            iso   = IsolationForest(contamination=.05,random_state=42)
            preds = iso.fit_predict(s.values.reshape(-1,1))
            out_if= (preds==-1).sum()
        rv   = ((s<EXPECTED_RANGES[col][0])|(s>EXPECTED_RANGES[col][1])).sum() if col in EXPECTED_RANGES else 0
        or_  = max(out_z,out_iq)/n
        st   = {"zscore_outliers":int(out_z),"iqr_outliers":int(out_iq),
                "isolation_forest_outliers":int(out_if),"range_violations":int(rv),
                "outlier_rate_pct":round(or_*100,2)}
        rg   = rate(or_,OUTLIER_WARN,OUTLIER_FAIL)
        fi   = (f"{out_z} Z-score outliers (|z|>3), {out_iq} IQR outliers, "
                f"{out_if} Isolation Forest anomalies ({or_*100:.1f}% of non-null).")
        rc   = ("Investigate extreme values vs source system. Consider Winsorization "
                "if outliers are data errors." if rg!="PASS"
                else "Outlier rates within acceptable bounds.")
    elif ct == "categorical":
        vc = s.value_counts(normalize=True); rare = (vc<.01).sum()
        st = {"unique_values":int(s.nunique()),"rare_categories_lt1pct":int(rare)}
        rg = "WARN" if rare>0 else "PASS"
        fi = f"{s.nunique()} categories; {rare} appear in <1% of records."
        rc = ("Review rare categories against data dictionary."
              if rare>0 else "All categories appear at reasonable frequencies.")
    elif ct == "datetime":
        future=(s>pd.Timestamp.now()).sum(); old=(s<pd.Timestamp("1900-01-01")).sum()
        st={"future_dates":int(future),"pre_1900_dates":int(old)}
        rg="FAIL" if (future+old)>0 else "PASS"
        fi=f"{future} future dates; {old} dates before 1900-01-01."
        rc=("Trace these to source and correct before model training."
            if rg!="PASS" else "No implausible dates detected.")
    else:
        rg,fi,rc="NA","",""
    return DQResult("Accuracy",col,ct,ok,rg,st,fi,rc)


# ── 2. AVAILABILITY ───────────────────────────────────────────────────────────
def assess_availability(df,col,ct):
    total=len(df[col]); nn=df[col].count(); null_n=total-nn
    fr=nn/total; nr=1-fr
    st={"total_records":int(total),"available_records":int(nn),
        "missing_records":int(null_n),"fill_rate_pct":round(fr*100,2),
        "null_rate_pct":round(nr*100,2)}
    rg=rate(nr,MISSING_WARN,MISSING_FAIL)
    fi=(f"Fill rate: {fr*100:.1f}% ({nn:,}/{total:,}). {null_n:,} records unavailable.")
    rc=("Investigate root cause: system dropout, ETL failure, or structural absence."
        if rg!="PASS" else "Data availability within acceptable thresholds.")
    return DQResult("Availability",col,ct,True,rg,st,fi,rc)


# ── 3. COMPLETENESS ───────────────────────────────────────────────────────────
def assess_completeness(df,col,ct):
    total=len(df[col]); null_n=df[col].isnull().sum(); nr=null_n/total
    blank=0
    if ct in ("categorical","text"):
        blank=(df[col].astype(str).str.strip()=="").sum()
    st={"null_count":int(null_n),"null_pct":round(nr*100,2),
        "blank_count":int(blank),"complete_pct":round((1-nr)*100,2)}
    rg=rate(nr,MISSING_WARN,MISSING_FAIL)
    fi=(f"Completeness: {(1-nr)*100:.1f}% — {null_n:,} nulls."
        +(f" {blank} blank strings also found." if blank>0 else ""))
    rc=("Determine if MCAR/MAR/MNAR. MNAR requires domain-specific treatment."
        if rg!="PASS" else "Completeness is satisfactory.")
    return DQResult("Completeness",col,ct,True,rg,st,fi,rc)


# ── 4. COMPREHENSIVENESS ──────────────────────────────────────────────────────
def assess_comprehensiveness(df,col,ct):
    ok,why=feasibility_check("Comprehensiveness",ct)
    if not ok: return DQResult("Comprehensiveness",col,ct,False,infeasible_reason=why)
    s=df[col].dropna(); st={}
    if ct=="categorical":
        ac=set(s.unique())
        if col in EXPECTED_CATEGORIES:
            ec=set(EXPECTED_CATEGORIES[col]); mc=ec-ac; xc=ac-ec
            cov=len(ac&ec)/len(ec) if ec else 1
            st={"expected":len(ec),"actual":len(ac),
                "missing_cats":str(list(mc)),"extra_cats":str(list(xc)),
                "coverage_pct":round(cov*100,2)}
            rg=rate(cov,.8,.6,False); fi=(f"Coverage: {cov*100:.1f}%. Missing: {mc or 'None'}.")
            rc=("Missing expected categories may appear in production."
                if rg!="PASS" else "All expected categories present.")
        else:
            vc=s.value_counts(normalize=True)
            gini=1-(vc**2).sum()
            st={"unique":int(s.nunique()),"gini_diversity":round(float(gini),4)}
            rg="PASS" if gini>.3 else "WARN"
            fi=f"{s.nunique()} categories. Gini: {gini:.3f}."
            rc="Specify EXPECTED_CATEGORIES for a complete coverage check."
    elif ct=="numeric":
        lo,hi=s.min(),s.max()
        if col in EXPECTED_RANGES:
            el,eh=EXPECTED_RANGES[col]; bel=(s<el).sum(); abo=(s>eh).sum()
            er=eh-el; oc=min(hi,eh)-max(lo,el); cr=max(0,oc)/er if er>0 else 1
            st={"obs_min":round(float(lo),4),"obs_max":round(float(hi),4),
                "exp_min":el,"exp_max":eh,"range_coverage_pct":round(cr*100,2)}
            rg=rate(cr,.8,.6,False)
            fi=f"Observed [{lo:.2f},{hi:.2f}] vs expected [{el},{eh}]. Coverage: {cr*100:.1f}%."
            rc=("Range does not fully span expected domain; model may extrapolate."
                if rg!="PASS" else "Observed range spans expected domain adequately.")
        else:
            p5,p95=s.quantile(.05),s.quantile(.95)
            st={"min":round(float(lo),4),"max":round(float(hi),4),
                "p5":round(float(p5),4),"p95":round(float(p95),4)}
            rg="PASS"; fi=f"Range [{lo:.4f},{hi:.4f}]. P5-P95: [{p5:.4f},{p95:.4f}]."
            rc="Specify EXPECTED_RANGES for domain coverage check."
    elif ct=="datetime":
        sd=pd.to_datetime(s); span=(sd.max()-sd.min()).days
        gaps=sd.sort_values().diff().dt.days.dropna()
        mg=gaps.max(); mg_m=gaps.mean()
        st={"span_days":int(span),"max_gap_days":round(float(mg),1),"mean_gap_days":round(float(mg_m),2)}
        rg="WARN" if mg>90 else "PASS"
        fi=f"Span: {span} days. Max gap: {mg:.0f} days. Mean gap: {mg_m:.1f} days."
        rc=("Large temporal gap may indicate missing periods. Verify extraction coverage."
            if rg!="PASS" else "Temporal coverage appears complete.")
    return DQResult("Comprehensiveness",col,ct,ok,rg,st,fi,rc)


# ── 5. CONSISTENCY ────────────────────────────────────────────────────────────
def assess_consistency(df,col,ct):
    ok,why=feasibility_check("Consistency",ct)
    if not ok: return DQResult("Consistency",col,ct,False,infeasible_reason=why)
    s=df[col].dropna(); st={}
    if ct=="numeric":
        mu=s.mean(); sd=s.std(); cv=sd/abs(mu) if mu!=0 else np.nan
        st={"mean":round(float(mu),4),"std":round(float(sd),4),
            "cv":round(float(cv),4) if not np.isnan(cv) else None}
        rg="WARN" if (cv is not None and cv>2) else "PASS"
        fi=(f"CV: {cv:.2f}. High CV (>2) may indicate unit inconsistencies or mixed populations."
            if cv is not None else "Mean ≈ 0; CV undefined.")
        rc=("Investigate whether variable was sourced from systems with different units."
            if rg!="PASS" else "Dispersion consistent with single-population source.")
    elif ct=="categorical":
        v=s.astype(str); lv=v.str.lower().unique(); av=v.unique()
        ci=len(av)-len(lv); ws=(v!=v.str.strip()).sum()
        st={"unique_raw":len(av),"unique_normalised":len(lv),
            "case_variants":max(0,ci),"whitespace_issues":int(ws)}
        rg="FAIL" if ci>0 or ws>0 else "PASS"
        fi=(f"Raw unique: {len(av)}; after normalise: {len(lv)}. "
            f"Case variants: {max(0,ci)}. Whitespace issues: {ws}.")
        rc=("Standardise casing and strip whitespace in ETL. "
            "Inconsistent encoding inflates cardinality and impairs model learning."
            if rg!="PASS" else "No encoding inconsistencies detected.")
    elif ct=="datetime":
        sd=pd.to_datetime(s); is_mon=(sd.sort_values().diff().dt.days.dropna()>=0).all()
        st={"is_monotonic":bool(is_mon),"min":str(sd.min().date()),"max":str(sd.max().date())}
        rg="PASS" if is_mon else "WARN"
        fi=(f"Date range {sd.min().date()} to {sd.max().date()}. "
            +("Monotonically increasing." if is_mon else "NOT monotonically increasing — ordering issue."))
        rc=("Investigate out-of-order dates; may indicate join errors."
            if not is_mon else "Temporal ordering is consistent.")
    return DQResult("Consistency",col,ct,ok,rg,st,fi,rc)


# ── 6. PLAUSIBILITY ───────────────────────────────────────────────────────────
def assess_plausibility(df,col,ct):
    ok,why=feasibility_check("Plausibility",ct)
    if not ok: return DQResult("Plausibility",col,ct,False,infeasible_reason=why)
    s=df[col].dropna(); st={}
    if ct=="numeric":
        Q1,Q3=s.quantile(.25),s.quantile(.75); IQR=Q3-Q1
        fl,fh=Q1-3*IQR,Q3+3*IQR; ex=((s<fl)|(s>fh)).sum(); er=ex/len(s)
        rv=((s<EXPECTED_RANGES[col][0])|(s>EXPECTED_RANGES[col][1])).sum() if col in EXPECTED_RANGES else 0
        st={"fence_lo":round(float(fl),4),"fence_hi":round(float(fh),4),
            "extreme_outliers":int(ex),"extreme_rate_pct":round(er*100,2),
            "domain_violations":int(rv)}
        rg=rate(er,.01,.03)
        fi=(f"Tukey(3×IQR) extremes: {ex} ({er*100:.2f}%). Fence: [{fl:.2f},{fh:.2f}]."
            +(f" Domain violations: {rv}." if col in EXPECTED_RANGES else ""))
        rc=("Verify extreme values against source. Apply robust model techniques if genuine."
            if rg!="PASS" else "No implausible extreme values detected.")
    elif ct=="categorical":
        vc=s.value_counts(normalize=True); imp=(vc<.001).sum()
        st={"total_cats":int(s.nunique()),"sub_0.1pct_cats":int(imp)}
        rg="WARN" if imp>0 else "PASS"
        fi=f"{imp} categories appear in <0.1% of records — potential data entry errors."
        rc=("Review rare categories vs data dictionary."
            if imp>0 else "All categories appear with plausible frequency.")
    elif ct=="datetime":
        sd=pd.to_datetime(s); fut=(sd>pd.Timestamp.now()).sum()
        old=(sd<pd.Timestamp("1970-01-01")).sum()
        st={"future_dates":int(fut),"pre_1970_dates":int(old)}
        rg=("FAIL" if (fut+old)/len(sd)>.01 else "WARN" if (fut+old)>0 else "PASS")
        fi=f"{fut} future dates; {old} dates before 1970-01-01."
        rc=("Future or very old dates are almost certainly erroneous."
            if rg!="PASS" else "All dates within plausible range.")
    return DQResult("Plausibility",col,ct,ok,rg,st,fi,rc)


# ── 7. REASONABLENESS ─────────────────────────────────────────────────────────
def assess_reasonableness(df,col,ct):
    ok,why=feasibility_check("Reasonableness",ct)
    if not ok: return DQResult("Reasonableness",col,ct,False,infeasible_reason=why)
    s=df[col].dropna(); st={}
    if ct=="numeric":
        p1,p5,p25,p50,p75,p95,p99=s.quantile([.01,.05,.25,.5,.75,.95,.99])
        iqr=p75-p25; wl,wh=p25-1.5*iqr,p75+1.5*iqr; op=((s<wl)|(s>wh)).mean()
        st={"p1":round(float(p1),4),"p5":round(float(p5),4),"p25":round(float(p25),4),
            "median":round(float(p50),4),"p75":round(float(p75),4),
            "p95":round(float(p95),4),"p99":round(float(p99),4),
            "iqr":round(float(iqr),4),"out_of_whisker_pct":round(op*100,2)}
        rg=rate(op,OUTLIER_WARN,OUTLIER_FAIL)
        fi=(f"IQR: {iqr:.4f}. Whiskers: [{wl:.4f},{wh:.4f}]. "
            f"{op*100:.2f}% outside 1.5×IQR. Median: {p50:.4f}.")
        rc=("Distribution tails heavier than expected. Validate extreme quantiles."
            if rg!="PASS" else "Value distribution within reasonable bounds.")
    elif ct=="categorical":
        vc=s.value_counts(normalize=True); H=float(scipy_entropy(vc.values,base=2))
        mH=np.log2(len(vc)) if len(vc)>1 else 1; nH=H/mH if mH>0 else 1
        dc=vc.index[0]; df_=vc.iloc[0]
        st={"shannon_entropy":round(H,4),"norm_entropy":round(nH,4),
            "dominant_cat":str(dc),"dominant_freq_pct":round(df_*100,2)}
        rg=("WARN" if df_>.95 else "WARN" if nH<.3 else "PASS")
        fi=(f"Shannon entropy: {H:.3f} bits (norm: {nH:.3f}). "
            f"Dominant '{dc}' at {df_*100:.1f}%.")
        rc=("Severe class imbalance — consider SMOTE or class-weighted training."
            if df_>.95 else "Moderate imbalance — monitor class distribution drift."
            if rg!="PASS" else "Class distribution is reasonably balanced.")
    elif ct=="datetime":
        sd=pd.to_datetime(s); yr=sd.dt.year.value_counts(normalize=True)
        cv_=yr.std()/yr.mean() if yr.mean()>0 else 0
        st={"year_cv":round(float(cv_),4),"top_year":int(yr.idxmax()),
            "top_year_pct":round(float(yr.max())*100,2)}
        rg="WARN" if cv_>1 else "PASS"
        fi=(f"Year CV: {cv_:.3f}. Most frequent year: {yr.idxmax()} ({yr.max()*100:.1f}%).")
        rc=("Uneven temporal distribution may introduce time-period bias."
            if rg!="PASS" else "Temporal distribution is reasonably uniform.")
    return DQResult("Reasonableness",col,ct,ok,rg,st,fi,rc)


# ── 8. REPLICABILITY ──────────────────────────────────────────────────────────
def assess_replicability(df,col,ct):
    col_hash=hashlib.md5(df[col].astype(str).str.cat().encode()).hexdigest()
    st={"column_md5":col_hash,"dataset_sha256":DATASET_HASH[:16]+"...",
        "record_count":int(df[col].count()),"unique_values":int(df[col].nunique())}
    fi=(f"Column MD5: {col_hash[:16]}... ({df[col].count():,} non-null, "
        f"{df[col].nunique():,} unique). Log this hash to detect upstream changes.")
    rc=("Store this checksum in your model governance system. "
        "Any change between runs indicates a data lineage event requiring re-validation.")
    return DQResult("Replicability",col,ct,True,"PASS",st,fi,rc)


# ── 9. REPRESENTATIVENESS ─────────────────────────────────────────────────────
def assess_representativeness(df,col,ct):
    ok,why=feasibility_check("Representativeness",ct)
    if not ok: return DQResult("Representativeness",col,ct,False,infeasible_reason=why)
    s=df[col].dropna(); st={}
    if ct=="numeric":
        sk=float(sp_skew(s)); ku=float(sp_kurtosis(s))
        ns,np_,nt=norm_test(s)
        st={"skewness":round(sk,4),"excess_kurtosis":round(ku,4),
            "normality_test":nt,"normality_stat":ns,"normality_p":np_}
        ask=abs(sk)
        rg=("FAIL" if ask>=SKEW_FAIL else "WARN" if ask>=SKEW_WARN else
            "WARN" if (np_ is not None and np_<.05) else "PASS")
        fi=(f"Skewness: {sk:.3f}. Kurtosis: {ku:.3f}. "
            +(f"{nt}: stat={ns:.4f}, p={np_:.4f}." if np_ is not None else ""))
        rc=("High skewness: sample may over-represent one tail. Consider log transform."
            if rg!="PASS" else "Distribution shape consistent with representative sample.")
    elif ct=="categorical":
        vc=s.value_counts(); c2,cp=stats.chisquare(vc.values)
        g=1-((vc/vc.sum())**2).sum()
        st={"chi2_uniform":round(float(c2),4),"chi2_p":round(float(cp),5),
            "gini":round(float(g),4),"n_cats":int(s.nunique())}
        rg="WARN" if cp<.05 else "PASS"
        fi=(f"Chi² uniformity: χ²={c2:.2f}, p={cp:.4f}. Gini: {g:.3f}. "
            +("Significantly non-uniform." if cp<.05 else "Cannot reject uniform."))
        rc=("Validate against known population proportions."
            if rg!="PASS" else "Category distribution consistent with representative sampling.")
    elif ct=="datetime":
        sd=pd.to_datetime(s); mo=sd.dt.to_period("M").value_counts()
        cv_=float(mo.std()/mo.mean()) if mo.mean()>0 else 0
        st={"months_repr":int(len(mo)),"monthly_cv":round(cv_,4),
            "min_month_n":int(mo.min()),"max_month_n":int(mo.max())}
        rg="WARN" if cv_>.5 else "PASS"
        fi=(f"{len(mo)} months represented. Monthly CV: {cv_:.3f}.")
        rc=("High monthly volatility may indicate seasonal sampling bias."
            if rg!="PASS" else "Temporal record distribution reasonably uniform.")
    return DQResult("Representativeness",col,ct,ok,rg,st,fi,rc)


# ── 10. TIMELINESS ────────────────────────────────────────────────────────────
def assess_timeliness(df,col,ct):
    ok,why=feasibility_check("Timeliness",ct)
    if not ok: return DQResult("Timeliness",col,ct,False,infeasible_reason=why)
    s=pd.to_datetime(df[col].dropna()); now=pd.Timestamp.now()
    lat=s.max(); old=s.min(); age=(now-lat).days; span=(lat-old).days
    sr=(s<now-pd.Timedelta(days=STALENESS_WARN)).mean()
    st={"latest_record":str(lat.date()),"oldest_record":str(old.date()),
        "age_days":int(age),"span_days":int(span),
        "stale_records_pct":round(float(sr)*100,2)}
    rg=rate(age,STALENESS_WARN,STALENESS_FAIL)
    fi=(f"Latest: {lat.date()} ({age} days ago). Oldest: {old.date()}. "
        f"Span: {span} days. {sr*100:.1f}% older than {STALENESS_WARN} days.")
    rc=(f"Data is {age} days stale. Refresh extract if deployment reflects recent conditions."
        if rg!="PASS" else "Data recency within acceptable thresholds.")
    return DQResult("Timeliness",col,ct,True,rg,st,fi,rc)


# ── 11. TRACEABILITY ──────────────────────────────────────────────────────────
def assess_traceability(df,col,ct):
    s=df[col].dropna(); tot=len(df); nu=s.nunique(); ur=nu/tot; nr=df[col].isnull().mean()
    is_id=(ur>CARDINALITY_HIGH and nr==0); part=(ur>CARDINALITY_HIGH and nr>0)
    st={"unique_values":int(nu),"uniqueness_ratio":round(ur,4),
        "null_rate":round(nr,4),"is_candidate_key":bool(is_id)}
    if col in ID_COLUMNS:
        di=df[col].dropna().duplicated().sum()
        st["id_duplicate_count"]=int(di)
        rg="FAIL" if di>0 else "PASS"
        fi=(f"Declared ID. Duplicate IDs: {di}. Null IDs: {df[col].isnull().sum()}. "
            f"Uniqueness ratio: {ur*100:.2f}%.")
        rc=("Duplicate/null IDs break record-level traceability. Investigate ETL joins."
            if rg!="PASS" else "ID column valid for traceability.")
    else:
        rg="PASS" if is_id else ("WARN" if part else "PASS")
        fi=(f"Uniqueness ratio: {ur*100:.2f}%. "
            +("Qualifies as candidate key." if is_id else "Standard attribute column."))
        rc=("Declare in ID_COLUMNS for full traceability assessment."
            if is_id else "Ensure column is traceable to source system record.")
    return DQResult("Traceability",col,ct,True,rg,st,fi,rc)


# ── 12. UNIQUENESS ────────────────────────────────────────────────────────────
def assess_uniqueness(df,col,ct):
    n=len(df); da=df.duplicated(keep=False).sum(); dv=df[col].duplicated(keep=False).sum()
    nu=df[col].nunique(); cr=nu/n; dr=da/n
    st={"total_rows":int(n),"dup_rows":int(da),"dup_values_col":int(dv),
        "unique_values":int(nu),"cardinality_ratio":round(cr,4),
        "dup_row_pct":round(dr*100,2)}
    rg=rate(dr,DUPLICATE_WARN,DUPLICATE_FAIL)
    fi=(f"Cardinality: {cr:.4f} ({nu:,}/{n:,}). "
        f"Full-row duplicates: {da} ({dr*100:.2f}%). "
        f"Duplicate values in column: {dv}.")
    rc=("Duplicate rows may indicate ETL fan-out or improper joins. "
        "Remove before training to prevent data leakage."
        if rg!="PASS" else "Uniqueness within acceptable thresholds.")
    return DQResult("Uniqueness",col,ct,True,rg,st,fi,rc)


# ── 13. VALIDITY ──────────────────────────────────────────────────────────────
def assess_validity(df,col,ct):
    s=df[col]; st={}
    if ct=="numeric":
        inf_c=np.isinf(pd.to_numeric(s,errors="coerce")).sum()
        neg_c=(pd.to_numeric(s,errors="coerce")<0).sum()
        par=pd.to_numeric(s,errors="coerce").notnull().sum()
        st={"inf_values":int(inf_c),"negative_values":int(neg_c),
            "parseable_pct":round(par/len(s)*100,2)}
        rg="FAIL" if inf_c>0 else ("WARN" if par<len(s)*.99 else "PASS")
        fi=f"Inf values: {inf_c}. Negatives: {neg_c}. Parseable: {par/len(s)*100:.1f}%."
        rc=("Remove infinite values — mathematically invalid model inputs."
            if inf_c>0 else "Verify negatives are valid for this domain."
            if neg_c>0 else "Numeric type conformance satisfactory.")
    elif ct=="categorical":
        if col in EXPECTED_CATEGORIES:
            exp=set(EXPECTED_CATEGORIES[col]); v=s.dropna().isin(exp).sum()
            inv=s.dropna().shape[0]-v; ir=inv/len(s)
            st={"valid":int(v),"invalid":int(inv),"invalid_rate_pct":round(ir*100,2)}
            rg=rate(ir,.01,.05)
            fi=f"{inv} values ({ir*100:.2f}%) outside expected set {sorted(exp)}."
            rc=("Correct invalid values or update data dictionary."
                if rg!="PASS" else "All values conform to expected category set.")
        else:
            nb=s.isnull().sum(); blk=(s.astype(str).str.strip()=="").sum()
            st={"null_count":int(nb),"blank_count":int(blk)}
            rg="WARN" if blk>0 else "PASS"
            fi=f"{nb} nulls; {blk} blank strings. No EXPECTED_CATEGORIES specified."
            rc="Specify EXPECTED_CATEGORIES for full validity assessment."
    elif ct=="datetime":
        sd=pd.to_datetime(s,errors="coerce"); pf=max(0,int(sd.isnull().sum()-s.isnull().sum()))
        fut=(sd>pd.Timestamp.now()).sum()
        st={"parse_failures":pf,"future_dates":int(fut),
            "parseable_pct":round((1-pf/max(len(s),1))*100,2)}
        rg="FAIL" if pf>0 else ("WARN" if fut>0 else "PASS")
        fi=f"{pf} parse failures. {fut} future dates."
        rc=("Standardise to ISO 8601 (YYYY-MM-DD)." if pf>0
            else "Date format is valid.")
    elif ct=="text":
        ne=(s.astype(str).str.strip()=="").sum(); ns=(s.astype(str).str.len()<3).sum()
        st={"empty_strings":int(ne),"very_short":int(ns)}
        rg="WARN" if (ne+ns)>0 else "PASS"
        fi=f"{ne} empty strings; {ns} very short (<3 chars)."
        rc="Implement domain-specific regex validation for this text field."
    else:
        rg,fi,rc,st="NA","","",{}
    return DQResult("Validity",col,ct,True,rg,st,fi,rc)


ASSESS_FN = {
    "Accuracy":assess_accuracy,"Availability":assess_availability,
    "Completeness":assess_completeness,"Comprehensiveness":assess_comprehensiveness,
    "Consistency":assess_consistency,"Plausibility":assess_plausibility,
    "Reasonableness":assess_reasonableness,"Replicability":assess_replicability,
    "Representativeness":assess_representativeness,"Timeliness":assess_timeliness,
    "Traceability":assess_traceability,"Uniqueness":assess_uniqueness,
    "Validity":assess_validity,
}
print("✅ All 13 assessment functions defined.")

In [ ]:
# ── Run all 13 × N assessments ────────────────────────────────────────────────
ALL_RESULTS = []
total = len(DIMENSIONS) * len(df.columns)
done  = 0

print(f"⚙️  Running {total} assessments "
      f"({len(DIMENSIONS)} dimensions × {len(df.columns)} columns)...\n")

for col in df.columns:
    ct = COL_TYPES[col]
    for dim, fn in ASSESS_FN.items():
        try:
            if dim=="Timeliness" and ct!="datetime":
                r = DQResult(dim,col,ct,False,
                             infeasible_reason=DIMENSIONS[dim]["infeasible_reason"].get(ct,""))
            else:
                r = fn(df, col, ct)
        except Exception as e:
            r = DQResult(dim,col,ct,True,"WARN",
                         findings=f"Assessment error: {str(e)[:120]}")
        ALL_RESULTS.append(r)
        done += 1
    pct = done/total*100
    print(f"  ✓ {col:<30} [{ct:<12}] — {pct:.0f}% complete")

results_df = pd.DataFrame([r.to_dict() for r in ALL_RESULTS])

RATING_ORDER = {"PASS":0,"WARN":1,"FAIL":2,"NA":3}
pivot = results_df.pivot_table(index="Column",columns="Dimension",
                               values="Rating",aggfunc="first")
pivot_num = pivot.replace(RATING_ORDER)

print(f"\n✅ Complete. {len(ALL_RESULTS):,} assessments | "
      f"Results table: {results_df.shape}")

In [ ]:
# ── Per-Variable Results in Notebook ─────────────────────────────────────────
display(Markdown("---\n# 📊 Per-Variable Assessment Results\n---"))

for col in df.columns:
    ct       = COL_TYPES[col]
    col_res  = [r for r in ALL_RESULTS if r.column==col]

    badge = " ".join([f"`{EMOJI[r.rating]} {r.dimension}: {r.rating}`"
                      for r in col_res])

    rows_md = []
    for r in col_res:
        stats_s = "; ".join([f"{k}={v}" for k,v in list(r.stats.items())[:4]])
        if not r.feasible:
            stats_s = f"*{r.infeasible_reason[:80]}*"
        fi_s = (r.findings[:110]+"..." if len(r.findings)>110 else r.findings)
        rows_md.append(
            f"| {EMOJI[r.rating]} **{r.rating}** | {r.dimension} | "
            f"{'✅' if r.feasible else '➖'} | {stats_s} | {fi_s} |"
        )

    table_md = (
        f"\n## 🔹 `{col}`  *(Type: {ct})*\n\n"
        + badge + "\n\n"
        + "| Rating | Dimension | Feasible | Key Stats | Key Finding |\n"
        + "|--------|-----------|:--------:|-----------|-------------|\n"
        + "\n".join(rows_md) + "\n\n---"
    )
    display(Markdown(table_md))

In [ ]:
# ── Column Visualization Engine ───────────────────────────────────────────────
VIZ_FIGURES = []

def make_col_fig(col, col_res_map):
    ct  = COL_TYPES[col]
    s   = df[col].dropna()
    nr  = df[col].isnull().mean()
    fig = plt.figure(figsize=(20,14), facecolor=PALETTE["bg"])
    fig.suptitle(f"Column: '{col}'  |  Type: {ct}  |  n={len(s):,}  |  null={nr*100:.1f}%",
                 fontsize=14, fontweight="bold", color=PALETTE["dark"], y=0.99)
    gs  = gridspec.GridSpec(3,4,figure=fig,hspace=0.55,wspace=0.40)

    # Panel A – distribution
    ax1 = fig.add_subplot(gs[0,:2]); ax1.set_facecolor("white")
    if ct=="numeric":
        ax1.hist(s,bins=min(50,int(np.sqrt(len(s)))),color=PALETTE["INFO"],
                 edgecolor="white",alpha=.85,density=True)
        try:
            xk=np.linspace(s.min(),s.max(),200)
            ax1.plot(xk,stats.gaussian_kde(s)(xk),color=PALETTE["dark"],lw=2,label="KDE")
        except: pass
        ax1.axvline(s.mean(),color=PALETTE["FAIL"],lw=1.5,ls="--",label=f"Mean={s.mean():.2f}")
        ax1.axvline(s.median(),color=PALETTE["PASS"],lw=1.5,ls="--",label=f"Med={s.median():.2f}")
        ax1.legend(fontsize=8); ax1.set_title("Histogram + KDE")
    elif ct=="categorical":
        vc=s.value_counts().head(15)
        ax1.barh(vc.index.astype(str),vc.values,color=PALETTE["INFO"],edgecolor="white")
        ax1.set_title(f"Top {len(vc)} Categories"); ax1.set_xlabel("Count")
    elif ct=="datetime":
        sd=pd.to_datetime(s)
        sd.groupby(sd.dt.to_period("M")).count().plot(ax=ax1,color=PALETTE["INFO"])
        ax1.set_title("Records per Month"); ax1.tick_params(axis="x",rotation=45)

    # Panel B – box / freq / yearly
    ax2=fig.add_subplot(gs[0,2:]); ax2.set_facecolor("white")
    if ct=="numeric":
        ax2.boxplot(s,vert=False,patch_artist=True,
                    boxprops=dict(facecolor=PALETTE["INFO"],alpha=.6),
                    medianprops=dict(color=PALETTE["FAIL"],lw=2),
                    flierprops=dict(marker="o",markersize=3,alpha=.4))
        ax2.set_title("Box Plot"); ax2.set_yticks([])
    elif ct=="categorical":
        vc2=s.value_counts(normalize=True)
        cx=[PALETTE["PASS"] if v>.05 else PALETTE["WARN"] if v>.01
            else PALETTE["FAIL"] for v in vc2.values]
        ax2.bar(range(len(vc2)),vc2.values,color=cx,edgecolor="white")
        ax2.set_xticks(range(len(vc2)))
        ax2.set_xticklabels(vc2.index.astype(str)[:20],rotation=45,ha="right",fontsize=8)
        ax2.set_title("Category Frequency (%)"); ax2.set_ylabel("Proportion")
    elif ct=="datetime":
        sd=pd.to_datetime(s); yr=sd.dt.year.value_counts().sort_index()
        ax2.bar(yr.index,yr.values,color=PALETTE["INFO"],edgecolor="white")
        ax2.set_title("Records per Year"); ax2.set_xlabel("Year")

    # Panel C – dimension rating heatmap
    ax3=fig.add_subplot(gs[1,:2]); ax3.set_facecolor("white")
    dl=[d for d in DIMENSIONS]; rl=[col_res_map[d].rating for d in dl]
    rn=[{"PASS":1,"WARN":2,"FAIL":3,"NA":0}[r] for r in rl]
    cm=matplotlib.colors.ListedColormap(["#ECF0F1",PALETTE["PASS"],
                                          PALETTE["WARN"],PALETTE["FAIL"]])
    ax3.imshow([rn],cmap=cm,vmin=0,vmax=3,aspect="auto")
    ax3.set_xticks(range(len(dl))); ax3.set_xticklabels(dl,rotation=45,ha="right",fontsize=8)
    ax3.set_yticks([])
    for i,(r,rnum) in enumerate(zip(rl,rn)):
        ax3.text(i,0,EMOJI[r],ha="center",va="center",fontsize=10)
    ax3.set_title("13-Dimension Rating Heatmap")

    # Panel D – QQ / pie / monthly
    ax4=fig.add_subplot(gs[1,2:]); ax4.set_facecolor("white")
    if ct=="numeric":
        probplot(s,plot=ax4); ax4.set_title("Q-Q Plot (Normality)")
    elif ct=="categorical":
        vc3=s.value_counts().head(8)
        ax4.pie(vc3.values,labels=vc3.index.astype(str),autopct="%1.1f%%",
                colors=sns.color_palette("husl",len(vc3))); ax4.set_title("Top 8 Share")
    elif ct=="datetime":
        sd=pd.to_datetime(s); mo=sd.dt.month.value_counts().sort_index()
        ax4.bar(mo.index,mo.values,color=PALETTE["accent"],edgecolor="white")
        ax4.set_xticks(range(1,13))
        ax4.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
        ax4.set_title("Seasonality")

    # Panel E – missing bar
    ax5=fig.add_subplot(gs[2,:2]); ax5.set_facecolor("white")
    ax5.barh(["Available","Missing"],[1-nr,nr],
             color=[PALETTE["PASS"],PALETTE["FAIL"]],edgecolor="white")
    ax5.set_xlim(0,1)
    ax5.axvline(MISSING_WARN,color="orange",ls="--",lw=1,label=f"WARN {MISSING_WARN*100:.0f}%")
    ax5.axvline(MISSING_FAIL,color="red",ls="--",lw=1,label=f"FAIL {MISSING_FAIL*100:.0f}%")
    ax5.legend(fontsize=8); ax5.set_title("Availability / Completeness"); ax5.set_xlabel("Proportion")

    # Panel F – stats text box
    ax6=fig.add_subplot(gs[2,2:]); ax6.axis("off")
    pc=sum(1 for r in col_res_map.values() if r.rating=="PASS")
    wc=sum(1 for r in col_res_map.values() if r.rating=="WARN")
    fc=sum(1 for r in col_res_map.values() if r.rating=="FAIL")
    nc=sum(1 for r in col_res_map.values() if r.rating=="NA")
    lines=[f"{'─'*34}",f"  Column : {col}",f"  Type   : {ct}",
           f"  N Total: {len(df[col]):,}",f"  N Valid: {len(s):,}",
           f"  Nulls  : {df[col].isnull().sum():,} ({nr*100:.1f}%)",
           f"  Unique : {s.nunique():,}"]
    if ct=="numeric":
        lines+=[f"  Mean   : {s.mean():.4f}",f"  Std    : {s.std():.4f}",
                f"  Min    : {s.min():.4f}",f"  Max    : {s.max():.4f}",
                f"  Skew   : {float(sp_skew(s)):.3f}",f"  Kurt   : {float(sp_kurtosis(s)):.3f}"]
    elif ct=="categorical":
        vc_=s.value_counts(); lines+=[f"  N Cats : {s.nunique()}",
                                        f"  Top    : '{vc_.index[0]}' ({vc_.iloc[0]:,})"]
    elif ct=="datetime":
        sd=pd.to_datetime(s)
        lines+=[f"  Min Dt : {sd.min().date()}",f"  Max Dt : {sd.max().date()}",
                f"  Span   : {(sd.max()-sd.min()).days} days"]
    lines+=[f"{'─'*34}",
            f"  ✅ PASS:{pc}  ⚠️ WARN:{wc}",
            f"  ❌ FAIL:{fc}  ➖ N/A :{nc}"]
    ax6.text(.05,.95,"\n".join(lines),transform=ax6.transAxes,fontsize=8.5,va="top",
             fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.5",facecolor=PALETTE["light"],
                       edgecolor=PALETTE["border"],alpha=.9))
    VIZ_FIGURES.append(fig)
    return fig

print("📈 Generating column visualizations...")
for col in df.columns:
    crm = {r.dimension:r for r in ALL_RESULTS if r.column==col}
    fig = make_col_fig(col, crm)
    plt.show()
    print(f"   ✓ {col}")
print("\n✅ All visualizations complete.")

In [ ]:
# ── Executive Scorecard & Radar Chart ─────────────────────────────────────────
display(Markdown("---\n# 🏆 Executive Scorecard\n---"))

dim_scores = {}
for dim in DIMENSIONS:
    dr = [r for r in ALL_RESULTS if r.dimension==dim and r.feasible]
    if not dr: continue
    pc=sum(1 for r in dr if r.rating=="PASS")
    wc=sum(1 for r in dr if r.rating=="WARN")
    fc=sum(1 for r in dr if r.rating=="FAIL")
    tot=pc+wc+fc; sc=(pc*3+wc*1)/(tot*3) if tot>0 else 0
    ag=("FAIL" if fc>0 else "WARN" if wc>0 else "PASS")
    dim_scores[dim]={"counts":{"PASS":pc,"WARN":wc,"FAIL":fc},
                     "total":tot,"score":sc,"rating":ag}

all_f=[r for r in ALL_RESULTS if r.feasible]
overall_score=(sum(3 for r in all_f if r.rating=="PASS")+
               sum(1 for r in all_f if r.rating=="WARN"))/(len(all_f)*3)*100
overall_rating=("FAIL" if overall_score<60 else "WARN" if overall_score<80 else "PASS")

# ── Figure: bar + radar ────────────────────────────────────────────────────────
fig_sc, axes = plt.subplots(1,2,figsize=(20,8),facecolor=PALETTE["bg"])
fig_sc.suptitle(f"{REPORT_TITLE} — Executive Scorecard",
                fontsize=15,fontweight="bold",color=PALETTE["dark"])

dsorted = sorted(dim_scores,key=lambda d:dim_scores[d]["score"],reverse=True)
spct    = [dim_scores[d]["score"]*100 for d in dsorted]
bc      = [PALETTE[dim_scores[d]["rating"]] for d in dsorted]

ax_b = axes[0]; ax_b.set_facecolor("white")
bars = ax_b.barh(dsorted,spct,color=bc,edgecolor="white",height=.65,alpha=.9)
ax_b.axvline(80,color="green",lw=1,ls="--",alpha=.7,label="80% (PASS)")
ax_b.axvline(60,color="orange",lw=1,ls="--",alpha=.7,label="60% (WARN)")
for bar,sc_ in zip(bars,spct):
    ax_b.text(bar.get_width()+1,bar.get_y()+bar.get_height()/2,
              f"{sc_:.1f}%",va="center",fontsize=9,fontweight="bold")
ax_b.set_xlim(0,115); ax_b.legend(fontsize=9)
ax_b.set_xlabel("Quality Score (%)"); ax_b.set_title("Dimension Quality Scores")

# Radar
axes[1].remove()
ax_r = fig_sc.add_subplot(1,2,2,projection="polar")
N=len(dim_scores); cats=list(dim_scores.keys())
vals=[dim_scores[d]["score"] for d in cats]+[dim_scores[cats[0]]["score"]]
ang=[n/float(N)*2*np.pi for n in range(N)]+[0]
ax_r.plot(ang,vals,"o-",lw=2,color=PALETTE["INFO"])
ax_r.fill(ang,vals,alpha=.25,color=PALETTE["INFO"])
ax_r.set_xticks(ang[:-1]); ax_r.set_xticklabels(cats,size=8,fontweight="bold")
ax_r.set_ylim(0,1); ax_r.set_yticks([.25,.5,.75,1])
ax_r.set_yticklabels(["25%","50%","75%","100%"],size=7)
ax_r.set_title("Quality Radar",size=13,fontweight="bold",pad=20)
plt.tight_layout()
VIZ_FIGURES.insert(0,fig_sc); plt.show()

# ── Heatmap ────────────────────────────────────────────────────────────────────
fig_hm, ax_hm = plt.subplots(
    figsize=(22,max(8,len(df.columns)*.55+2)), facecolor=PALETTE["bg"])
cmap = matplotlib.colors.ListedColormap(
    [PALETTE["NA"],PALETTE["PASS"],PALETTE["WARN"],PALETTE["FAIL"]])
ax_hm.imshow(pivot_num.values,cmap=cmap,vmin=0,vmax=3,aspect="auto")
ax_hm.set_xticks(range(len(pivot_num.columns)))
ax_hm.set_xticklabels(pivot_num.columns,rotation=40,ha="right",fontsize=9,fontweight="bold")
ax_hm.set_yticks(range(len(pivot_num.index)))
ax_hm.set_yticklabels(pivot_num.index,fontsize=9)
for i in range(len(pivot_num.index)):
    for j in range(len(pivot_num.columns)):
        r_=pivot.iloc[i,j]; ax_hm.text(j,i,EMOJI.get(r_,"➖"),ha="center",va="center",fontsize=10)
ax_hm.legend(handles=[mpatches.Patch(color=PALETTE[r],label=r) for r in ["PASS","WARN","FAIL","NA"]],
             loc="upper left",bbox_to_anchor=(1.01,1),fontsize=10)
ax_hm.set_title(f"Data Quality Heatmap — All Columns × All Dimensions",
                fontsize=14,fontweight="bold",color=PALETTE["dark"])
plt.tight_layout(); VIZ_FIGURES.insert(1,fig_hm); plt.show()

# ── Markdown summary ───────────────────────────────────────────────────────────
md_r=["| Dimension | PASS | WARN | FAIL | Score | Rating |",
      "|-----------|:----:|:----:|:----:|------:|--------|"]
for d in dsorted:
    c_=dim_scores[d]["counts"]; sc_=dim_scores[d]["score"]; rt_=dim_scores[d]["rating"]
    md_r.append(f"| **{d}** | {c_['PASS']} | {c_['WARN']} | {c_['FAIL']} "
                f"| {sc_*100:.1f}% | {EMOJI[rt_]} **{rt_}** |")
display(Markdown(
    "## 📈 Dimension Summary\n\n" + "\n".join(md_r)
    + f"\n\n> **Overall DQ Score: {overall_score:.1f}%** — "
    + f"{EMOJI[overall_rating]} **{overall_rating}**"))
print(f"\n🎯 Overall Score: {overall_score:.1f}% [{overall_rating}]")

In [ ]:
# ── Generate PDF Report ────────────────────────────────────────────────────────
print(f"📄 Generating PDF: {OUTPUT_PDF} ...")

with PdfPages(OUTPUT_PDF) as pdf:

    # Cover page
    fig_c,ax_c=plt.subplots(figsize=(11,8.5)); fig_c.patch.set_facecolor(PALETTE["dark"])
    ax_c.axis("off")
    for text,y,fs,fw,color in [
        (REPORT_TITLE, .80,22,"bold","white"),
        (REPORT_SUBTITLE,.71,12,"normal","#BDC3C7"),
        (f"Model: {MODEL_NAME}",.63,11,"normal","#95A5A6"),
        (f"Dataset: {DATASET_PATH}  |  {NROWS:,} rows × {NCOLS} cols",.55,10,"normal","#95A5A6"),
        (f"SHA-256: {DATASET_HASH[:32]}...",.47,8,"normal","#7F8C8D"),
        (f"Author: {REPORT_AUTHOR}  |  {LOAD_TIMESTAMP.strftime('%Y-%m-%d %H:%M')}",.12,9,"normal","#7F8C8D"),
    ]:
        ax_c.text(.5,y,text,ha="center",fontsize=fs,fontweight=fw,color=color,
                  transform=ax_c.transAxes,
                  fontfamily=("monospace" if "SHA" in text else "DejaVu Sans"))
    ax_c.add_patch(FancyBboxPatch((.25,.29),.5,.10,boxstyle="round,pad=.02",
                   facecolor=PALETTE[overall_rating],transform=ax_c.transAxes))
    ax_c.text(.5,.34,f"Overall DQ Score: {overall_score:.1f}%  [{overall_rating}]",
              ha="center",va="center",fontsize=14,fontweight="bold",color="white",
              transform=ax_c.transAxes)
    pdf.savefig(fig_c,bbox_inches="tight"); plt.close(fig_c)

    # Scorecard + heatmap
    for fig in VIZ_FIGURES[:2]:
        pdf.savefig(fig,bbox_inches="tight")

    # Column figures
    for fig in VIZ_FIGURES[2:]:
        pdf.savefig(fig,bbox_inches="tight")

    # Issues register pages
    fw_ = results_df[results_df["Rating"].isin(["FAIL","WARN"])].sort_values(
        ["Rating","Column"]).reset_index(drop=True)
    rpp=16
    for ps in range(0,len(fw_),rpp):
        chunk=fw_.iloc[ps:ps+rpp]
        fig_t,ax_t=plt.subplots(figsize=(22,11),facecolor=PALETTE["bg"]); ax_t.axis("off")
        ax_t.set_title(f"Issues Register — FAIL & WARN (page {ps//rpp+1}/"
                       f"{-(-len(fw_)//rpp)})",fontsize=13,fontweight="bold",
                       color=PALETTE["dark"])
        cw=[.10,.13,.07,.06,.34,.30]; hds=["Column","Dimension","Type","Rating","Finding","Recommendation"]
        xp=[sum(cw[:i]) for i in range(len(cw))]
        yh=.93
        for xv,hd in zip(xp,hds):
            ax_t.text(xv,yh,hd,fontsize=9,fontweight="bold",color="white",va="top",
                      bbox=dict(facecolor=PALETTE["dark"],pad=3,edgecolor="none"),
                      transform=ax_t.transAxes)
        for ri,(idx,row) in enumerate(chunk.iterrows()):
            y=yh-(ri+1)*.87/rpp
            ax_t.axhline(y+.005,color=PALETTE["border"],lw=.5,alpha=.5,
                         transform=ax_t.transAxes)
            vals=[row["Column"],row["Dimension"],row["Type"],row["Rating"],
                  textwrap.fill(str(row["Findings"])[:100],55),
                  textwrap.fill(str(row["Recommendation"])[:90],48)]
            for xv,v in zip(xp,vals):
                c_=PALETTE.get(str(v),PALETTE["dark"]) if v in ("FAIL","WARN") else PALETTE["dark"]
                fw__="bold" if v in ("FAIL","WARN") else "normal"
                ax_t.text(xv,y,str(v),fontsize=7.5,va="top",
                          color=c_,fontweight=fw__,transform=ax_t.transAxes)
        pdf.savefig(fig_t,bbox_inches="tight"); plt.close(fig_t)

    d=pdf.infodict()
    d["Title"]=REPORT_TITLE; d["Author"]=REPORT_AUTHOR
    d["Subject"]="SR 11-7 Model Risk Data Quality Validation"

print(f"✅ PDF saved: {OUTPUT_PDF}")

In [ ]:
# ── Generate Excel Report ──────────────────────────────────────────────────────
print(f"📊 Generating Excel: {OUTPUT_EXCEL} ...")
wb = openpyxl.Workbook()

F={k:PatternFill("solid",fgColor=v.lstrip("#")) for k,v in {
    "PASS":"27AE60","WARN":"E67E22","FAIL":"E74C3C",
    "NA":"95A5A6","header":"2C3E50","sub":"34495E"}.items()}
FW=Font(color="FFFFFF",bold=True,size=10)
FB=Font(bold=True,size=10); FN=Font(size=9)
CA=Alignment(horizontal="center",vertical="center",wrap_text=True)
LA=Alignment(horizontal="left",vertical="top",wrap_text=True)
THN=Border(left=Side(style="thin",color="CCCCCC"),right=Side(style="thin",color="CCCCCC"),
           top=Side(style="thin",color="CCCCCC"),bottom=Side(style="thin",color="CCCCCC"))

def hdr(ws,cols,row=1):
    for j,(cn,w) in enumerate(cols,1):
        c=ws.cell(row=row,column=j,value=cn)
        c.fill=F["header"]; c.font=FW; c.alignment=CA; c.border=THN
        ws.column_dimensions[get_column_letter(j)].width=w

# Sheet 1 – Executive Summary
ws1=wb.active; ws1.title="Executive Summary"
ws1.merge_cells("A1:H1")
h=ws1["A1"]
h.value=f"📋  {REPORT_TITLE}  |  Score: {overall_score:.1f}%  [{overall_rating}]"
h.fill=F[overall_rating]; h.font=FW; h.alignment=CA; ws1.row_dimensions[1].height=28
for i,(k,v) in enumerate([("Dataset",DATASET_PATH),("Model",MODEL_NAME),
                            ("Author",REPORT_AUTHOR),("Generated",str(LOAD_TIMESTAMP.date())),
                            ("Rows",f"{NROWS:,}"),("Columns",NCOLS),
                            ("SHA-256",DATASET_HASH[:32]+"...")],3):
    ws1.cell(i,1,k).font=FB; ws1.cell(i,2,str(v)).font=FN
ws1.cell(12,1,"Dimension Summary").font=FB
hdr(ws1,[("Dimension",22),("PASS",8),("WARN",8),("FAIL",8),
         ("Score%",10),("Rating",10),("Definition",50)],13)
dsorted2=sorted(dim_scores,key=lambda d:dim_scores[d]["score"],reverse=True)
for i,d in enumerate(dsorted2,14):
    c_=dim_scores[d]["counts"]; sc_=dim_scores[d]["score"]; rt_=dim_scores[d]["rating"]
    vals=[d,c_["PASS"],c_["WARN"],c_["FAIL"],f"{sc_*100:.1f}%",rt_,
          DIMENSIONS[d]["definition"][:150]]
    for j,v in enumerate(vals,1):
        cell=ws1.cell(i,j,v); cell.border=THN; cell.alignment=LA
        if j==6: cell.fill=F[rt_]; cell.font=FW; cell.alignment=CA

# Sheet 2 – Full Results
ws2=wb.create_sheet("Full Results")
hdr(ws2,[("Column",22),("Type",12),("Dimension",22),("Feasible",10),
         ("Rating",10),("Key Stats",45),("Findings",70),("Recommendation",55)])
for i,row in enumerate(results_df.itertuples(),2):
    ss="; ".join([f"{k.replace('stat_','')}={v}" for k,v in row._asdict().items()
                  if k.startswith("stat_") and pd.notna(v)])[:120]
    vals=[row.Column,row.Type,row.Dimension,row.Feasible,row.Rating,
          ss,str(row.Findings)[:200],str(row.Recommendation)[:180]]
    for j,v in enumerate(vals,1):
        c=ws2.cell(i,j,v); c.border=THN; c.alignment=LA; c.font=FN
        if j==5 and str(v) in F: c.fill=F[str(v)]; c.font=FW; c.alignment=CA

# Sheet 3 – Issues Register
ws3=wb.create_sheet("Issues Register")
iss=results_df[results_df["Rating"].isin(["FAIL","WARN"])].sort_values(
    ["Rating","Column"]).reset_index(drop=True)
hdr(ws3,[("Priority",12),("Column",22),("Type",12),("Dimension",22),
         ("Rating",10),("Findings",80),("Recommendation",60)])
for i,row in enumerate(iss.itertuples(),2):
    p="P1-Critical" if row.Rating=="FAIL" else "P2-Major"
    vals=[p,row.Column,row.Type,row.Dimension,row.Rating,
          str(row.Findings)[:300],str(row.Recommendation)[:200]]
    for j,v in enumerate(vals,1):
        c=ws3.cell(i,j,v); c.border=THN; c.alignment=LA; c.font=FN
        if j in (1,5) and str(row.Rating) in F:
            c.fill=F[str(row.Rating)]; c.font=FW; c.alignment=CA

# Sheet 4 – Heatmap
ws4=wb.create_sheet("Rating Heatmap")
pv2=pivot.reset_index()
hdr(ws4,[("Column",22)]+[(d,16) for d in pivot.columns])
for i,row in enumerate(pv2.itertuples(index=False),2):
    for j,v in enumerate(list(row),1):
        c=ws4.cell(i,j,v); c.border=THN; c.alignment=CA
        if j>1 and str(v) in F:
            c.fill=F[str(v)]; c.font=Font(color="FFFFFF",bold=True,size=9)
        elif j==1: c.font=FB

wb.save(OUTPUT_EXCEL)
print(f"✅ Excel saved: {OUTPUT_EXCEL}")

In [ ]:
# ── Final Summary ──────────────────────────────────────────────────────────────
all_f=[r for r in ALL_RESULTS if r.feasible]
n_p=sum(1 for r in all_f if r.rating=="PASS")
n_w=sum(1 for r in all_f if r.rating=="WARN")
n_f=sum(1 for r in all_f if r.rating=="FAIL")
n_na=sum(1 for r in ALL_RESULTS if r.rating=="NA")

display(Markdown(f"""
---
## ✅ Assessment Complete

| Metric | Value |
|--------|-------|
| **Overall DQ Score** | **{overall_score:.1f}%** |
| **Overall Rating** | **{EMOJI[overall_rating]} {overall_rating}** |
| Total Assessments | {len(ALL_RESULTS):,} |
| Feasible Assessments | {len(all_f):,} |
| ✅ PASS | {n_p} ({n_p/max(len(all_f),1)*100:.1f}%) |
| ⚠️ WARN | {n_w} ({n_w/max(len(all_f),1)*100:.1f}%) |
| ❌ FAIL | {n_f} ({n_f/max(len(all_f),1)*100:.1f}%) |
| ➖ N/A  | {n_na} |
| **PDF Report** | `{OUTPUT_PDF}` |
| **Excel Report** | `{OUTPUT_EXCEL}` |
| Dataset SHA-256 | `{DATASET_HASH[:32]}...` |
| Timestamp | {LOAD_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S')} |
"""))

fails=[(r.column,r.dimension,r.findings[:130]) for r in ALL_RESULTS if r.rating=="FAIL"]
if fails:
    rows_f="\n".join(f"| `{c}` | {d} | {f} |" for c,d,f in fails[:25])
    display(Markdown(
        "### 🔴 FAIL Findings\n\n"
        "| Column | Dimension | Finding |\n"
        "|--------|-----------|---------|\n" + rows_f))
else:
    display(Markdown("*No FAIL findings — excellent data quality!*"))

print(f"\n📁 Outputs: {OUTPUT_PDF}  |  {OUTPUT_EXCEL}") 